# Notebook 2: Document Processing & Vector Stores

**Learning Goal**: Load PDF documents, split them into chunks, create vector embeddings, store them in FAISS, and retrieve relevant documents by semantic similarity.

**rag_engine.py coverage**: This notebook covers `load_and_index_docs()` (lines 16-43), `format_docs()` (lines 45-46), and the retriever setup (lines 55-57).

---

## 1. Document Loading

### The Problem

LLMs can only process **text**. But real-world knowledge lives in PDFs, Word docs, web pages, and databases. **Document loaders** bridge this gap by extracting text from various file formats.

### The Document Object

LangChain represents loaded content as `Document` objects with two attributes:
- **`page_content`**: The extracted text
- **`metadata`**: Information about the source (file name, page number, etc.)

### PyPDFLoader

For PDF files, LangChain provides `PyPDFLoader`. It creates one `Document` per page.

> **In `rag_engine.py`** (lines 28-31): Each PDF in the `docs/` folder is loaded with `PyPDFLoader`, and all documents are combined into a single list.

In [ ]:
# Install required packages (skip if already installed)
# %pip install langchain langchain-openai langchain-community faiss-cpu pypdf python-dotenv

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()

from langchain_community.document_loaders import PyPDFLoader

# Load a single PDF file
# Using a small PDF from the docs/ folder for demonstration
loader = PyPDFLoader(os.path.join("..", "docs", "NHIF_Vifurushi_Bima_Afya_2025.pdf"))
documents = loader.load()

print(f"Number of pages loaded: {len(documents)}")
print(f"Type of each item: {type(documents[0])}")

In [ ]:
# Inspect the first document
doc = documents[0]

print("=== METADATA ===")
print(doc.metadata)

print("\n=== PAGE CONTENT (first 500 chars) ===")
print(doc.page_content[:500])

### Loading Multiple PDFs

In production, we load **all** PDFs from a folder. Here's how `rag_engine.py` does it:

```python
# rag_engine.py lines 20-31
documents = []
files = [f for f in os.listdir(docs_folder) if f.endswith('.pdf')]
for file in files:
    file_path = os.path.join(docs_folder, file)
    loader = PyPDFLoader(file_path)
    documents.extend(loader.load())
```

In [ ]:
# Load all PDFs from the docs folder (same pattern as rag_engine.py)
docs_folder = os.path.join("..", "docs")
all_documents = []

files = [f for f in os.listdir(docs_folder) if f.endswith('.pdf')]
print(f"Found {len(files)} PDF files")

for file in files:
    file_path = os.path.join(docs_folder, file)
    loader = PyPDFLoader(file_path)
    loaded = loader.load()
    all_documents.extend(loaded)
    print(f"  {file}: {len(loaded)} pages")

print(f"\nTotal documents (pages): {len(all_documents)}")

## 2. Text Splitting

### Why Split Documents?

LLMs have a limited **context window** (the amount of text they can process at once). A single PDF page might be thousands of characters, and we may have hundreds of pages. We can't send them all to the LLM.

Instead, we:
1. **Split** documents into small **chunks**
2. **Search** for the most relevant chunks when a user asks a question
3. **Send only those chunks** to the LLM as context

### RecursiveCharacterTextSplitter

This is LangChain's recommended text splitter. It tries to keep semantically related text together by splitting on natural boundaries in this order:
1. Double newlines (`\n\n`) — paragraph breaks
2. Single newlines (`\n`) — line breaks
3. Spaces (` `) — word boundaries
4. Characters — last resort

Key parameters:
- **`chunk_size`**: Maximum characters per chunk
- **`chunk_overlap`**: Characters shared between consecutive chunks (prevents losing context at boundaries)

> **In `rag_engine.py`** (lines 33-34): `chunk_size=1000, chunk_overlap=200` — each chunk is up to 1000 chars, with 200 chars of overlap to maintain continuity.

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Same settings as rag_engine.py line 33
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

# Split the documents into chunks
splits = text_splitter.split_documents(documents)  # using the single PDF we loaded earlier

print(f"Original pages: {len(documents)}")
print(f"After splitting: {len(splits)} chunks")

In [ ]:
# Inspect a few chunks
for i, chunk in enumerate(splits[:3]):
    print(f"--- Chunk {i} ({len(chunk.page_content)} chars) ---")
    print(chunk.page_content[:200] + "...")
    print(f"Metadata: {chunk.metadata}")
    print()

### Visualizing Chunk Overlap

Let's see how `chunk_overlap` works. The end of one chunk should overlap with the beginning of the next:

In [ ]:
if len(splits) >= 2:
    chunk_0_end = splits[0].page_content[-100:]  # last 100 chars of chunk 0
    chunk_1_start = splits[1].page_content[:100]  # first 100 chars of chunk 1

    print("=== End of Chunk 0 ===")
    print(chunk_0_end)
    print("\n=== Start of Chunk 1 ===")
    print(chunk_1_start)
    print("\n(Notice the overlapping text between the two chunks)")
else:
    print("Not enough chunks to demonstrate overlap. Try with a longer document.")

### Experimenting with Chunk Size

The chunk size is a trade-off:
- **Too small** → Chunks may lack context, losing meaning
- **Too large** → Retrieval becomes less precise, noise increases

A common starting point is **500-1500 characters** with **10-20% overlap**.

In [ ]:
# Compare different chunk sizes
for size in [500, 1000, 2000]:
    splitter = RecursiveCharacterTextSplitter(chunk_size=size, chunk_overlap=int(size * 0.2))
    chunks = splitter.split_documents(documents)
    print(f"chunk_size={size:>4}, overlap={int(size*0.2):>3} -> {len(chunks)} chunks")

## 3. Embeddings

### Theory: From Text to Vectors

**Embeddings** convert text into numerical vectors (lists of numbers). These vectors capture the **semantic meaning** of the text. Similar texts produce similar vectors.

```
"What is health insurance?"  →  [0.023, -0.041, 0.089, ...]
"Tell me about medical coverage"  →  [0.021, -0.038, 0.092, ...]  # similar!
"What's the weather today?"  →  [-0.056, 0.073, -0.012, ...]  # very different
```

This is what makes **semantic search** possible — we can find relevant documents even when the user's question uses different words than the source text.

> **In `rag_engine.py`** (line 39): `OpenAIEmbeddings()` uses OpenAI's embedding model (`text-embedding-ada-002` by default).

In [ ]:
from langchain_openai import OpenAIEmbeddings

# Initialize the embedding model — same as rag_engine.py line 39
embeddings = OpenAIEmbeddings()

# Embed a single text
vector = embeddings.embed_query("What is health insurance?")

print(f"Vector dimension: {len(vector)}")
print(f"First 10 values: {vector[:10]}")

In [ ]:
import numpy as np

# Demonstrate semantic similarity with cosine similarity
texts = [
    "What is health insurance?",
    "Tell me about medical coverage",
    "What's the weather today?",
]

vectors = [embeddings.embed_query(text) for text in texts]

def cosine_similarity(a, b):
    a, b = np.array(a), np.array(b)
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

print("Cosine Similarity (1.0 = identical, 0.0 = unrelated):")
print(f"  'health insurance' vs 'medical coverage':  {cosine_similarity(vectors[0], vectors[1]):.4f}")
print(f"  'health insurance' vs 'weather today':     {cosine_similarity(vectors[0], vectors[2]):.4f}")

## 4. Vector Stores (FAISS)

### Theory: Why Vector Stores?

Once we have embeddings for all our document chunks, we need a way to:
1. **Store** thousands of vectors efficiently
2. **Search** for the most similar vectors quickly

**FAISS** (Facebook AI Similarity Search) is a library optimized for fast similarity search. It uses **approximate nearest neighbor** algorithms to find similar vectors in milliseconds, even across millions of entries.

### Workflow
```
Documents → Text Splitter → Chunks → Embeddings → FAISS Index
                                                       ↑
User Query → Embedding → Similarity Search ────────────┘
                              ↓
                    Top-K Similar Chunks
```

> **In `rag_engine.py`** (lines 40-41): `FAISS.from_documents()` creates the index, and `save_local()` persists it to disk.

In [ ]:
from langchain_community.vectorstores import FAISS

# Create a FAISS vector store from our document chunks
# This embeds all chunks and stores them — same as rag_engine.py line 40
vectorstore = FAISS.from_documents(splits, embeddings)

print(f"Vector store created with {len(splits)} chunks")

In [ ]:
# Search for similar documents (basic similarity search)
query = "What are the NHIF benefit packages?"
results = vectorstore.similarity_search(query, k=3)

print(f"Query: '{query}'")
print(f"Found {len(results)} results:\n")

for i, doc in enumerate(results):
    print(f"--- Result {i+1} (source: {doc.metadata.get('source', 'unknown')}, page: {doc.metadata.get('page', '?')}) ---")
    print(doc.page_content[:300] + "...")
    print()

### Saving and Loading the Index

Creating embeddings costs time and API calls. FAISS lets you **save** the index to disk and **reload** it later without re-embedding.

> **In `rag_engine.py`**:
> - Line 41: `vectorstore.save_local(FAISS_INDEX_PATH)` — saves after indexing
> - Line 56: `FAISS.load_local(...)` — loads for querying

In [ ]:
import tempfile

# Save the index to a temporary directory (production uses "faiss_index")
save_path = os.path.join(tempfile.gettempdir(), "tutorial_faiss_index")
vectorstore.save_local(save_path)
print(f"Index saved to: {save_path}")

# Reload the index from disk
loaded_vectorstore = FAISS.load_local(
    save_path,
    embeddings,
    allow_dangerous_deserialization=True  # required for loading pickled data
)
print("Index loaded successfully!")

# Verify it works
results = loaded_vectorstore.similarity_search("benefit packages", k=1)
print(f"Search test: found {len(results)} result(s)")

## 5. Retrieval

A **retriever** is a LangChain abstraction that wraps a vector store and returns relevant documents for a query. It's what connects the vector store to the rest of the chain.

Key parameter:
- **`k`**: Number of documents to return. Higher k = more context but also more noise.

> **In `rag_engine.py`** (line 57): `vectorstore.as_retriever(search_kwargs={"k": 10})` — returns the top 10 most relevant chunks per query.

In [ ]:
# Convert vector store to a retriever — same as rag_engine.py line 57
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})  # using k=3 for demo

# Invoke the retriever
docs = retriever.invoke("What services are covered by NHIF?")

print(f"Retrieved {len(docs)} documents:\n")
for i, doc in enumerate(docs):
    print(f"--- Document {i+1} ---")
    print(doc.page_content[:200] + "...")
    print()

### The `format_docs` Helper

When passing retrieved documents to an LLM prompt, we need to convert them from a list of `Document` objects to a single string. The `format_docs` function does this.

> **In `rag_engine.py`** (lines 45-46):
> ```python
> def format_docs(docs):
>     return "\n\n".join(doc.page_content for doc in docs)
> ```

In [ ]:
# The format_docs helper — same as rag_engine.py lines 45-46
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# Use it to format retrieved documents
formatted = format_docs(docs)
print(f"Formatted context ({len(formatted)} chars):")
print(formatted[:500] + "...")

## 6. Putting It Together: The `load_and_index_docs` Function

Now we can understand the full `load_and_index_docs()` function from `rag_engine.py` (lines 16-43). It combines everything we've learned:

```
PDF files → PyPDFLoader → Documents → RecursiveCharacterTextSplitter → Chunks
    → OpenAIEmbeddings → FAISS.from_documents → save_local
```

### Production Code Reference
```python
# rag_engine.py lines 16-43
def load_and_index_docs(docs_folder):
    documents = []
    files = [f for f in os.listdir(docs_folder) if f.endswith('.pdf')]
    for file in files:
        file_path = os.path.join(docs_folder, file)
        loader = PyPDFLoader(file_path)
        documents.extend(loader.load())
    
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
    splits = text_splitter.split_documents(documents)
    
    embeddings = OpenAIEmbeddings()
    vectorstore = FAISS.from_documents(splits, embeddings)
    vectorstore.save_local(FAISS_INDEX_PATH)
    
    return f"Indexed {len(documents)} documents into {len(splits)} chunks."
```

In [ ]:
# Let's rebuild it step by step with commentary

# Step 1: Load all PDFs
docs_folder = os.path.join("..", "docs")
documents = []
files = [f for f in os.listdir(docs_folder) if f.endswith('.pdf')]
for file in files:
    loader = PyPDFLoader(os.path.join(docs_folder, file))
    documents.extend(loader.load())
print(f"Step 1 - Loaded {len(documents)} pages from {len(files)} PDFs")

# Step 2: Split into chunks
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
splits = text_splitter.split_documents(documents)
print(f"Step 2 - Split into {len(splits)} chunks")

# Step 3: Create embeddings and FAISS index
embeddings = OpenAIEmbeddings()
vectorstore = FAISS.from_documents(splits, embeddings)
print(f"Step 3 - Created FAISS index with {len(splits)} vectors")

# Step 4: Save to disk
tutorial_index_path = os.path.join(tempfile.gettempdir(), "tutorial_full_index")
vectorstore.save_local(tutorial_index_path)
print(f"Step 4 - Saved index to {tutorial_index_path}")

print(f"\nDone! Indexed {len(documents)} documents into {len(splits)} chunks.")

## Summary

In this notebook we learned:

| Concept | What we learned | rag_engine.py reference |
|---|---|---|
| `PyPDFLoader` | Load text from PDF files | Lines 30-31 |
| `Document` | Text + metadata container | Lines 20, 31 |
| `RecursiveCharacterTextSplitter` | Split text into semantic chunks | Lines 33-34 |
| `OpenAIEmbeddings` | Convert text to vectors | Line 39 |
| `FAISS` | Store and search vectors | Lines 40-41, 56 |
| `.as_retriever()` | Create a retriever from a vector store | Line 57 |
| `format_docs()` | Convert document list to string | Lines 45-46 |

**Next notebook**: We'll combine retrieval with the LLM to build the complete RAG chain with conversation history support.